# STEP 2 - Quantum Computing: ownership, concentration, entry and subfields

Questions of this step
1. How concentrated is Quantum Computing patent ownership?
2. Consolidating or not?
3. Who are the incumbents and who are the entrants?
4. Which subfields are growing faster?
5. What external forces can shape the field? (PEST)

We keep analysing **DOCDB patent families**, imported from Step 0.

**Name harmonisation problem.** The same company appears under slightly different names or with its subsidiaries. From `tls206_person` we can get
- `psn_id`, `psn_name`: PATSTAT standardised names
- `han_id`, `han_name`: OECD harmonised applicant names (better, but not complete)

Methods compared on the course gold standard (all implemented in `name_harmonisation.py`):

| code | method |
|---|---|
| M0 | identity (raw `psn_name`) |
| M1 | dictionary of legal forms (professor's method A) |
| M2 | `cleanco` + M3 |
| M3 | normalisation (dots, accents, `&`, brackets, `c/o`) + legal-form dictionary |
| M4 | M3 + fuzzy string grouping |
| M5 | **brand key**: first word of the cleaned name (two words for `UNIV X`, `QUANTUM X`, `INTERNATIONAL X` ...) |
| M5b | brand key + merge of keys that differ by a typo |
| M6 | Terrorizer-style: matching score, similarity graph, Louvain communities |
| M7 | clusters produced by the authors' official Terrorizer code (optional file) |
| +aliases | any of the above + a manual alias table for the top applicants |

**Order of the notebook:** config -> harmonisation (gold standard) -> data from PATSTAT -> harmonise applicants -> concentration -> consolidation over time -> incumbents/entrants -> subfields -> PEST -> files saved.

In [ ]:
import json
import re
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from matplotlib.ticker import PercentFormatter
from scipy import stats
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS206_PERSON,              # persons: standardised name, harmonised name, sector, country
    TLS207_PERS_APPLN,          # person <-> application links (applicant / inventor)
    TLS224_APPLN_CPC,           # CPC classification
)

## 0. Configuration

**All paths start from one place (`ROOT`)** and are printed, so you always see where files are read from and written to.
Step 0 wrote `outputs/step0/qc_families_<ENV>.csv`; this notebook writes everything to `outputs/step2/<ENV>/`.

In [ ]:
ENV = "PROD"                          # TEST / PROD. MUST be the same as in Step 0 (it is part of the file name)

# ---- paths: one anchor, everything else is derived ----
ROOT = Path.cwd()                     # the folder the notebook runs from
print("working directory:", ROOT)

SECTOR_FILE = ROOT / "outputs" / "step0" / f"qc_families_{ENV}.csv"      # written by Step 0 (cell "save the families")
OUTPUT_DIR = ROOT / "outputs" / "step2" / ENV                            # TEST and PROD never overwrite each other
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALIASES_FILE = ROOT / "outputs" / "step2" / "manual_aliases.csv"         # optional: columns alias,parent (see section 1.5)

GOLD_STANDARD_FILE = ROOT / "innovation-management-datascience26" / "tutorial_1" / "PATSTAT_500_label_datav4.csv"
EXTERNAL_CLUSTERS_FILE = None         # optional: output of the official Terrorizer code (csv/parquet), see section 1.4
EXTERNAL_NAME_COL, EXTERNAL_CLUSTER_COL = "name", "cluster"

# ---- which families are "Quantum Computing"? ----
SECTOR_NAME = "Quantum Computing"
SECTOR_RULE = "all"                   # "all" = every family of Step 0 (IPC | CPC | keywords);  "classified" = only IPC or CPC
# SAME CPC list as Step 0 (G06N10/, G06N99/002, Y10S977/933), otherwise subfields lose the older families
SECTOR_CPC_PREFIXES = ("G06N10/", "G06N99/002", "Y10S977/933")

# ---- time ----
YEAR_MIN, YEAR_MAX = 1990, 2023       # same as START_YEAR, END_YEAR in Step 0
LAST_COMPLETE_YEAR = 2020             # later years are still incomplete (publication lag)
IPR_TYPE = "PI"
SPLIT_YEAR = 2015                     # incumbent: first sector family before this year (justify this choice in the report)
WINDOW = 3                            # years per window for the concentration trend
MIN_FAMILIES_PER_WINDOW = 30          # earlier windows with fewer families are too noisy to interpret
TOP_N = 15

# ---- name harmonisation ----
NAME_METHOD = "brand_fuzzy"           # identity | dictionary | dictionary_v2 | cleanco | fuzzy | brand | brand_fuzzy | terrorizer | external
EMBED = "tfidf"                       # Terrorizer-style cosine: "tfidf" (offline) or "sbert" (sentence-transformers, as in the paper)
RUN_TUNING = True                     # tune the Terrorizer-style parameters on the gold standard (train/test split by group)
SECTOR_GENERIC_WORDS = ("QUANTUM", "COMPUTING", "QUBIT")   # words that must NOT make two names "the same company" in this sector
TERR_PARAMS = {"extra_generic": SECTOR_GENERIC_WORDS}      # used by brand and terrorizer; the tuning cell adds tuned weights

# ---- context for the PEST section: events drawn as vertical lines on the entry/exit chart ----
EVENTS = {}                           # e.g. {2018: "US National Quantum Initiative Act"}  <- verify and cite every event you add

COUNTRY_NAMES = {
    "AU": "Australia", "CA": "Canada", "CN": "China", "FR": "France",
    "DE": "Germany", "IL": "Israel", "IT": "Italy", "JP": "Japan",
    "NL": "Netherlands", "KR": "South Korea", "SE": "Sweden",
    "CH": "Switzerland", "GB": "UK", "US": "USA", "EP": "EPO",
}

In [ ]:
patstat = PatstatClient(env=ENV)
db = patstat.orm()

In [ ]:
# ---- the harmonisation module: put name_harmonisation.py next to this notebook (or anywhere under ROOT) ----
try:
    import name_harmonisation as nh
except ImportError:
    hits = list(ROOT.rglob("name_harmonisation.py"))
    if not hits:
        raise ImportError("name_harmonisation.py not found under " + str(ROOT) + ". Copy it next to this notebook.")
    sys.path.insert(0, str(hits[0].parent))
    import name_harmonisation as nh
print("using", nh.__file__)

try:
    import cleanco                                   # optional, only for method M2 / NAME_METHOD="cleanco"
except ImportError:
    print("cleanco not installed. For M2 run   %pip install cleanco   in a separate cell.")


def answer_box(question, headline, detail=""):
    "Small highlighted box for the answer to each question of the step."
    display(HTML(
        f'<div style="border-left:5px solid #2b6cb0;background:#eef4fb;padding:10px 14px;margin:10px 0">'
        f'<div style="font-weight:600">{question}</div><div style="margin-top:4px">{headline}</div>'
        f'<div style="margin-top:4px;color:#444;font-size:0.92em">{detail}</div></div>'))


def save_table(df, name, index=True):
    "Save a table in OUTPUT_DIR as csv (opens correctly in Excel) and say where."
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print("saved:", path)
    return path


def save_fig(fig, name):
    path = OUTPUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved:", path)
    return path

## 1. Name harmonisation

### 1.1 Quick look at the methods on a few names

In [ ]:
examples = ["SIEMENS AG", "Siemens Aktiengesellschaft", "BALLARD POWER SYSTEMS INC.", "Ballard Power Systems",
            "TOYOTA JIDOSHA KABUSHIKI KAISHA", "KABUSHIKI KAISHA TOSHIBA", "Toyota Motor Corp",
            "FIAT S.p.A.", "Umicore N.V.", "NOKIA MOBILE PHONES (U.K.) LIMITED", "Danfoss A/S", "SA"]

demo = pd.DataFrame({"name": examples})
for m in ["dictionary", "dictionary_v2", "brand_fuzzy", "terrorizer"]:
    # (brand_fuzzy and terrorizer see all the example names together)
    mapping = nh.harmonise_names(examples, method=m, embed=EMBED)
    demo[m] = demo["name"].map(mapping)
demo

Reading the table: `dictionary` (professor's method A) leaves `FIAT S P A` and `UMICORE N V`, because it splits on the dots before looking for legal forms;
`dictionary_v2` removes dots first, drops brackets / `c/o` parts and keeps a name such as `SA` instead of turning it into an empty string.
Neither can link `TOYOTA JIDOSHA` and `TOYOTA MOTOR`; the brand key does (both become `TOYOTA`), because it groups everything under the first word.

### 1.2 Evaluation on the gold standard

In [ ]:
if not GOLD_STANDARD_FILE.exists():                      # fall back to a search under ROOT
    found = list(ROOT.rglob("PATSTAT_500_label_data*.csv"))
    if not found:
        raise FileNotFoundError(f"Gold standard not found: {GOLD_STANDARD_FILE}")
    GOLD_STANDARD_FILE = found[0]
print("gold standard:", GOLD_STANDARD_FILE)

gold = pd.read_csv(GOLD_STANDARD_FILE).rename(columns={"comp": "name", "Group": "gold_group"})
print(f"{len(gold):,} names in {gold['gold_group'].nunique():,} organisations")

EXTERNAL_MAP = None
if EXTERNAL_CLUSTERS_FILE:
    EXTERNAL_MAP = nh.load_external_clusters(EXTERNAL_CLUSTERS_FILE, EXTERNAL_NAME_COL, EXTERNAL_CLUSTER_COL)
    print(f"loaded {len(EXTERNAL_MAP):,} names from the external Terrorizer output")

gold_results = nh.evaluate(gold, nh.gold_methods(embed=EMBED, params=TERR_PARAMS, external_map=EXTERNAL_MAP))
save_table(gold_results, "harmonisation_gold_results")
gold_results

**What the course gold standard looks like and what it shows** (3,192 names, 481 organisations; my run on your file):

| method | groups found | precision | recall | F1 |
|---|---|---|---|---|
| M0 identity | 3,192 | 1.000 | 0.000 | 0.000 |
| M1 dictionary (prof A) | 3,007 | 0.995 | 0.002 | 0.005 |
| M3 normalise + dictionary | 2,895 | 0.996 | 0.006 | 0.012 |
| M4 fuzzy 0.90 | 2,624 | 0.996 | 0.013 | 0.026 |
| **M5b brand key + typo merge** | 580 | 0.968 | 0.931 | **0.949** |
| M6 Terrorizer-style (no web data) | 1,284 | 0.910 | 0.185 | 0.308 |

The gold standard groups **a company with all its subsidiaries under the brand** (198 `SIEMENS ...` names are one organisation, and the 10 largest organisations hold 75% of all gold pairs).
Cleaning legal forms cannot reach that: `SIEMENS RAIL AUTOMATION LIMITED` and `SIEMENS HEALTHCARE LIMITED` are different strings after any cleaning. That is why M1-M4 have almost no recall, and why the pairwise F1 is dominated by a few giants.
The brand key reaches F1 0.95 because the gold standard was built by brand. Two cautions: (1) its rules were refined after looking at this gold standard, so the 0.95 is optimistic for new data; (2) the gold file contains only large multinationals and their variants, whereas your applicant list also has universities, start-ups and individuals, which is why the brand key has safeguards (two words for `UNIV TEXAS`, `QUANTUM CIRCUITS`, `INTERNATIONAL BUSINESS`, `D WAVE`) and why you must check the top of your own list by hand.
The Terrorizer-style method is far behind because the paper's decisive signal, the web domain / page text of each name, is missing here: without it the graph splits `SIEMENS AUTOMOTIVE` from `SIEMENS POWER`.

Metrics are pairwise: **precision** = of the pairs of names you merged, how many belong together in the gold standard; **recall** = of the pairs that belong together, how many you merged.
`homogeneity / completeness / V` are the cluster-level counterparts.

### 1.3 Tuning the Terrorizer-style method

The paper tunes the weights, the matching threshold and the Louvain resolution to maximise F1 (Optuna / TPE).
Here a random search does the same job without extra packages. On the course gold standard the gain is small (the missing web data is the limit), so `RUN_TUNING = False` is a reasonable setting once you have seen this. The gold standard is split **by organisation** into train and test halves, so the *test F1* is not inflated by tuning on the same names.

In [ ]:
if RUN_TUNING:
    TERR_PARAMS, tuning = nh.tune_terrorizer(gold, n_trials=150, embed=EMBED, extra_generic=SECTOR_GENERIC_WORDS)
    display(tuning)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in TERR_PARAMS.items()})
    save_table(tuning, "terrorizer_tuning")
    (OUTPUT_DIR / "terrorizer_params.json").write_text(json.dumps(TERR_PARAMS, indent=2, default=list))
else:
    print("tuning skipped, using the defaults:", {**nh.TERR_DEFAULTS, **TERR_PARAMS})

### 1.4 Terrorizer: what is implemented and what is not

`terrorizer` in this notebook follows the matching and filtering stages of Ascione & Sterzi (arXiv 2403.12083): a pairwise score from *any token in common*, *first token in common* and the *cosine similarity* of the names, an edge when the score reaches a threshold, Louvain community detection, and the most frequent name of each community as its label.
Words that name a *kind* of organisation rather than a specific one (`UNIV`, `INST`, `TECH`, country names, ... plus `SECTOR_GENERIC_WORDS`, here `QUANTUM`, `COMPUTING`) do not count as a shared token: otherwise `UNIV X` and `UNIV Y`, or `QUANTUM CIRCUITS` and `QUANTUM MACHINES`, would be merged.
It does **not** include the web "knowledge augmentation" (DuckDuckGo domain and page text), the bridgeness-centrality pruning or the location bonus, so results will differ from the authors' tool.
If you run their original code on the list of unique names, save its output as a csv with one column for the raw name and one for the cluster, set `EXTERNAL_CLUSTERS_FILE` in the config and use `NAME_METHOD = "external"`: it is compared in the table above as M7.

### 1.5 Look at the mistakes, and the manual alias table

In [ ]:
CHECK_METHOD = "brand_fuzzy"           # method whose errors you want to inspect
check_map = nh.harmonise_names(gold["name"], method=CHECK_METHOD, params=TERR_PARAMS, embed=EMBED, external_map=EXTERNAL_MAP)
pred = gold["name"].map(check_map).tolist()

print("OVER-MERGED: one predicted group contains several gold organisations (precision errors)")
display(nh.errors(gold, pred, "fp", n=15))
print("SPLIT: one gold organisation is spread over several predicted groups (recall errors)")
display(nh.errors(gold, pred, "fn", n=15).drop(columns=["example"]))

What the brand key gets wrong on the gold standard: joint ventures (`SONY ERICSSON`, `SUMITOMO MITSUBISHI`), brands whose subsidiaries start with a city (`SHENZHEN HUAWEI`, `GUANGDONG MIDEA`), acronyms (`EXXON` vs `EXXONMOBIL`), and French / Italian descriptive words (`DE TECHNOLGIE`, `CAVI`). These are exactly the cases to look for in your own top applicants.

**Manual aliases.** Concentration depends on the top of the applicant list, so correct that by hand.
After the first run, section 6 exports the top applicants with the raw name variants merged into each. Look for parents and subsidiaries that are still separate
(e.g. a university system under two spellings, a firm and its quantum subsidiary) and write them into `outputs/step2/manual_aliases.csv`:

```
alias,parent
TOYOTA MOTOR,TOYOTA JIDOSHA
UNIV CALIFORNIA,REGENTS OF THE UNIVERSITY OF CALIFORNIA
```

Both columns are cleaned the same way as the names, so you can paste raw PATSTAT names. An alias is matched first against the whole cleaned **raw name** (use it to merge or split a single name, e.g. `TEXAS INSTRUMENTS INC -> TEXAS INSTRUMENTS` takes it out of the `TEXAS` brand) and then against the **resulting applicant** (use it to merge whole groups, e.g. `EXXON -> EXXONMOBIL`). Re-run from section 4 and the aliases are applied on top of whatever method you chose. Keep the file: it is part of your method and should be described in the report.

## 2. Measuring concentration

At two levels: whole sector and country. Six measures from the literature:
- 4-firm and 8-firm concentration ratio
- Herfindahl index (and its numbers-equivalent 1/HH)
- Coefficient of variation
- Gini coefficient
- Lotka coefficient

The Lorenz curve sorts the firms from the smallest to the largest and plots the cumulative share of firms (x) against the cumulative share of patents (y).
With equal firms it is the diagonal; the more concentrated the patents, the further it bends below.
The Gini coefficient is the area between the diagonal and the curve, divided by the whole area below the diagonal.

In [ ]:
def concentration_ratio(patents_per_firm, k):
    "Share (%) of all patents held by the k largest firms."
    largest_first = np.sort(patents_per_firm)[::-1]
    return largest_first[:k].sum() / largest_first.sum() * 100


def herfindahl(patents_per_firm):
    "Sum of the squared shares of the firms."
    shares = np.asarray(patents_per_firm) / np.sum(patents_per_firm)
    return np.sum(shares ** 2)


def coefficient_of_variation(patents_per_firm):
    "Standard deviation of patents per firm divided by their mean."
    x = np.asarray(patents_per_firm, dtype=float)
    return x.std(ddof=1) / x.mean() if len(x) > 1 else np.nan


def lorenz_curve(patents_per_firm):
    "Cumulative share of firms (smallest first) and cumulative share of their patents."
    x = np.sort(patents_per_firm)
    share_of_firms = np.arange(0, len(x) + 1) / len(x)
    share_of_patents = np.concatenate([[0], np.cumsum(x) / x.sum()])
    return share_of_firms, share_of_patents


def gini(patents_per_firm):
    "Area between the diagonal and the Lorenz curve, divided by the area below the diagonal (= 0.5)."
    share_of_firms, share_of_patents = lorenz_curve(patents_per_firm)
    widths = np.diff(share_of_firms)
    heights = (share_of_patents[1:] + share_of_patents[:-1]) / 2
    return (0.5 - np.sum(widths * heights)) / 0.5


def lotka(patents_per_firm):
    "Slope (and its p-value) of log(number of firms with n patents) on log(n)."
    f = pd.Series(patents_per_firm).value_counts().sort_index()
    if len(f) < 3:
        return np.nan, np.nan
    fit = stats.linregress(np.log(f.index), np.log(f.values))
    return fit.slope, fit.pvalue


def six_measures(patents_per_firm):
    "All the measures for one group of firms, as one row of a table."
    lotka_slope, lotka_pvalue = lotka(patents_per_firm)
    return {
        "firms": len(patents_per_firm),
        "patents (sum over firms)": int(np.sum(patents_per_firm)),
        "CV": coefficient_of_variation(patents_per_firm),
        "CR4 (%)": concentration_ratio(patents_per_firm, 4),
        "CR8 (%)": concentration_ratio(patents_per_firm, 8),
        "HH": herfindahl(patents_per_firm),
        "1/HH": 1 / herfindahl(patents_per_firm),
        "Lotka L": lotka_slope,
        "Lotka p-value": lotka_pvalue,
        "Gini": gini(patents_per_firm),
    }


# quick check on two toy sectors: 10 equal firms, and one giant with 9 small firms
pd.DataFrame({"10 equal firms": six_measures([5] * 10),
              "1 giant + 9 small": six_measures([91] + [1] * 9)}).round(3)

## 3. Quantum Computing: data from PATSTAT

In [ ]:
def load_sector_ids(path, rule="all"):
    "Family IDs of Step 0. rule='classified' keeps only the families found through IPC or CPC (no keyword-only ones)."
    if not path.exists():
        print("Step 0 file not found:", path)
        print("candidates under", ROOT, ":")
        for p in nh.find_step0_file([ROOT])[:10]:
            print("  ", p)
        raise FileNotFoundError(f"{path}. Check ENV, or set SECTOR_FILE to one of the files above.")
    frame = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    id_cols = [c for c in frame.columns if "docdb_family_id" in str(c).lower()]
    if not id_cols:
        raise ValueError(f"Cannot identify the family-ID column. Columns: {list(frame.columns)}")
    frame["_id"] = pd.to_numeric(frame[id_cols[0]], errors="coerce")
    frame = frame.dropna(subset=["_id"])

    flags = [c for c in frame.columns if str(c).upper() in ("IPC", "CPC")]
    if flags:
        classified = frame[flags].astype(bool).any(axis=1)
        print(f"{len(frame):,} families in the file: {classified.sum():,} classified (IPC or CPC), "
              f"{(~classified).sum():,} found by keywords only")
        if rule == "classified":
            frame = frame[classified]
    elif rule == "classified":
        raise ValueError(f"SECTOR_RULE='classified' needs IPC / CPC columns. Columns: {list(frame.columns)}")
    return frame["_id"].astype("int64").unique().tolist()


sector_ids_all = load_sector_ids(SECTOR_FILE, SECTOR_RULE)
print(f"SECTOR_RULE={SECTOR_RULE!r}: {len(sector_ids_all):,} unique family IDs from {SECTOR_FILE}")

In [ ]:
ID_CHUNK = 5000                       # IDs per query: a very long IN (...) list can make the SQL too large


def family_scope(id_chunk):
    "One row per DOCDB family (of this chunk) with its earliest filing year; IPR type and window applied here."
    family_year = func.min(TLS201_APPLN.earliest_filing_year)
    return (
        db.query(TLS201_APPLN.docdb_family_id.label("unit_id"), family_year.label("year"))
        .filter(TLS201_APPLN.docdb_family_id.in_(id_chunk), TLS201_APPLN.ipr_type == IPR_TYPE)
        .group_by(TLS201_APPLN.docdb_family_id)
        .having(family_year.between(YEAR_MIN, YEAR_MAX))
        .subquery()
    )


def query_applicants(families):
    "Applicants of each family. applt_seq_nr > 0 keeps applicants and drops inventors."
    return (
        db.query(
            families.c.unit_id, families.c.year,
            TLS206_PERSON.psn_id, TLS206_PERSON.psn_name, TLS206_PERSON.psn_sector,
            TLS206_PERSON.han_id, TLS206_PERSON.han_name,                      # OECD harmonised name, for the robustness check
            func.trim(TLS206_PERSON.person_ctry_code).label("ctry"),
        )
        .select_from(families)
        .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
        .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
        .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)
        .distinct()
    )


def query_cpc(families):
    "CPC codes of the families, for the subfield analysis."
    return (
        db.query(families.c.unit_id, families.c.year,
                 func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("cpc"))
        .select_from(families)
        .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
        .join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
        .distinct()
    )


raw_parts, cpc_parts = [], []
for k in range(0, len(sector_ids_all), ID_CHUNK):
    scope = family_scope(sector_ids_all[k:k + ID_CHUNK])
    raw_parts.append(patstat.df(query_applicants(scope)))
    cpc_parts.append(patstat.df(query_cpc(scope)))
    print(f"  chunk {k // ID_CHUNK + 1}/{-(-len(sector_ids_all) // ID_CHUNK)} done")

raw_sector = pd.concat(raw_parts, ignore_index=True).drop_duplicates()
family_cpc = pd.concat(cpc_parts, ignore_index=True).drop_duplicates()

if raw_sector.empty:
    raise ValueError("No applicant-family rows returned. Check sector IDs, IPR_TYPE and years.")
if family_cpc.empty:
    raise ValueError("No CPC rows returned for the selected families.")

print(f"{raw_sector['unit_id'].nunique():,} families with applicant links")
print(f"{raw_sector['psn_id'].nunique():,} distinct PATSTAT persons, {raw_sector['psn_name'].nunique():,} distinct psn_name, "
      f"{raw_sector['han_name'].nunique():,} distinct han_name")

In [ ]:
# the first patent ever (in any field) of each applicant, to identify newcomers to patenting
psn_ids = [int(i) for i in raw_sector["psn_id"].unique()]
first_patents = []
for start in range(0, len(psn_ids), 5000):                      # in chunks, to keep each query small
    q_first = (
        db.query(TLS206_PERSON.psn_id, func.min(TLS201_APPLN.earliest_filing_year).label("first_patent_year"))
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
        .join(TLS201_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
        .filter(TLS206_PERSON.psn_id.in_(psn_ids[start:start + 5000]),
                TLS207_PERS_APPLN.applt_seq_nr > 0,
                TLS201_APPLN.earliest_filing_year.between(1900, YEAR_MAX))   # drops the 9999 placeholders
        .group_by(TLS206_PERSON.psn_id)
    )
    first_patents.append(patstat.df(q_first))
first_patents = (pd.concat(first_patents, ignore_index=True) if first_patents
                 else pd.DataFrame(columns=["psn_id", "first_patent_year"]))
print(f"first patent year found for {first_patents['psn_id'].nunique():,} of {len(psn_ids):,} persons")

## 4. Harmonise the applicants

`applicant_labels` gives every row of `raw_sector` its harmonised applicant, with the method chosen in the config (`NAME_METHOD`) plus the manual aliases (if the file exists).
With the brand methods, **check the top of the list for false merges** before trusting any concentration number: a brand shared by unrelated organisations (the notebook cannot know that two `PHOENIX ...` are different companies) inflates that applicant, and a joint venture is attached to its first partner.
Persons with no name at all are kept apart (`UNKNOWN <psn_id>`), so they do not merge into one giant applicant.

In [ ]:
ALIASES = nh.load_aliases(ALIASES_FILE) if ALIASES_FILE.exists() else {}
print(f"{len(ALIASES)} manual aliases loaded" if ALIASES else f"no alias file yet ({ALIASES_FILE})")

PSN_FAMILIES = raw_sector.groupby("psn_name")["unit_id"].nunique().to_dict()    # used to name Terrorizer communities


def applicant_labels(raw, how, aliases=ALIASES):
    "Harmonised applicant name for each row of raw. how: a method of name_harmonisation, or 'han' (OECD harmonised name)."
    names = raw["psn_name"].fillna("")
    if how == "han":
        han = raw["han_name"].fillna("").astype(str).str.strip()
        fallback = names.map(nh.clean_dictionary_v2)                       # no han_name: dictionary-cleaned psn_name
        labels = han.where(han != "", fallback)
    else:
        mapping = nh.harmonise_names(names.unique(), method=how, counts=PSN_FAMILIES, aliases=aliases,
                                     params=TERR_PARAMS, embed=EMBED, external_map=EXTERNAL_MAP)
        labels = names.map(mapping)
    blank = labels.fillna("").eq("")
    return labels.where(~blank, "UNKNOWN " + raw["psn_id"].astype(str))


def applicant_type(psn_sector):
    "Normalise PATSTAT psn_sector."
    if pd.isna(psn_sector):
        return "unknown"
    s = str(psn_sector).upper()
    if "UNIVERSITY" in s:
        return "university"
    if "COMPANY" in s:
        return "firm"
    if "GOV" in s:
        return "government / non-profit"
    if "HOSPITAL" in s:
        return "hospital"
    if "INDIVIDUAL" in s:
        return "individual"
    return "unknown"


def most_frequent(s):
    s = s.dropna()
    return s.value_counts().index[0] if len(s) else np.nan


def prepare_applicants(labelled):
    "From rows with an 'applicant' column to one row per (family, applicant) with type and country."
    df = labelled.copy()
    df["type"] = df["psn_sector"].map(applicant_type)
    known_type = df[df["type"] != "unknown"].groupby("applicant")["type"].agg(most_frequent)
    df["type"] = df["applicant"].map(known_type).fillna("unknown")                    # one type per applicant

    df["ctry"] = df["ctry"].replace("", np.nan)
    known = df.dropna(subset=["ctry"]).drop_duplicates(["unit_id", "applicant", "ctry"])
    main_country = known.groupby("applicant")["ctry"].agg(most_frequent)
    df["ctry"] = df["applicant"].map(main_country)                                     # one country per applicant
    return df[["unit_id", "year", "applicant", "type", "ctry"]].drop_duplicates(["unit_id", "applicant"])


labelled = raw_sector.assign(applicant=applicant_labels(raw_sector, NAME_METHOD))
links = prepare_applicants(labelled)

# first patent ever of each harmonised applicant: the earliest among the PATSTAT persons it groups
psn = labelled[["psn_id", "applicant"]].drop_duplicates().merge(first_patents, on="psn_id")
first_ever = psn.groupby("applicant")["first_patent_year"].min()

print(f"NAME_METHOD={NAME_METHOD!r}: {raw_sector['psn_name'].nunique():,} PATSTAT names -> {links['applicant'].nunique():,} applicants "
      f"on {links['unit_id'].nunique():,} families")

## 5. How concentrated is the sector?

Only complete years are used (`year <= LAST_COMPLETE_YEAR`). The count is **families per applicant** (an applicant counts once per family).

In [ ]:
complete = links[links["year"] <= LAST_COMPLETE_YEAR]
families_per_applicant = complete.groupby("applicant")["unit_id"].nunique()
firms_per_applicant = complete[complete["type"] == "firm"].groupby("applicant")["unit_id"].nunique()

sector_table = pd.DataFrame({
    f"all applicants ({NAME_METHOD})": six_measures(families_per_applicant.to_numpy()),
    "firms only": six_measures(firms_per_applicant.to_numpy()),
})
sector_table.loc["Lotka p-value"] = sector_table.loc["Lotka p-value"].astype(float)
save_table(sector_table.round(4), "concentration_sector")
sector_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for label, series in [("all applicants", families_per_applicant), ("firms only", firms_per_applicant)]:
    x, y = lorenz_curve(series.to_numpy())
    ax.plot(x, y, label=f"{label} (Gini {gini(series.to_numpy()):.2f})")
ax.plot([0, 1], [0, 1], color="grey", lw=0.8, ls="--")
ax.set(title=f"{SECTOR_NAME}: Lorenz curve of families per applicant", xlabel="cumulative share of applicants (smallest first)",
       ylabel="cumulative share of families")
ax.legend()
save_fig(fig, "lorenz_curve")
plt.show()

### Does the answer depend on how names are harmonised?

The same measures under several harmonisations, **including the OECD `han_name`**. If the conclusion (concentrated or not) is the same in every row, the result is robust to the harmonisation choice - worth one sentence in the report.

In [ ]:
rows = {}
for how in ["identity", "dictionary_v2", "brand", "brand_fuzzy", "terrorizer", "han"]:
    lab = prepare_applicants(raw_sector.assign(applicant=applicant_labels(raw_sector, how)))
    fpa = lab[lab["year"] <= LAST_COMPLETE_YEAR].groupby("applicant")["unit_id"].nunique().to_numpy()
    rows[how] = {k: v for k, v in six_measures(fpa).items() if k in ("firms", "CR4 (%)", "CR8 (%)", "HH", "1/HH", "Gini")}
robust = pd.DataFrame(rows).T
save_table(robust.round(4), "concentration_robustness_to_harmonisation")
robust.round(3)

### Country level

The same measures inside each of the largest countries (applicants are assigned to the country where they most often appear).

In [ ]:
country_rows = {}
top_countries = complete.groupby("ctry")["unit_id"].nunique().sort_values(ascending=False).head(8).index
for c in top_countries:
    fpa = complete[complete["ctry"] == c].groupby("applicant")["unit_id"].nunique().to_numpy()
    if len(fpa) >= 5:                                    # too few applicants: measures meaningless
        country_rows[COUNTRY_NAMES.get(c, c)] = six_measures(fpa)
country_table = pd.DataFrame(country_rows).T
save_table(country_table.round(4), "concentration_by_country")
country_table.round(3)

In [ ]:
m = sector_table.iloc[:, 0]
answer_box(
    "How concentrated is Quantum Computing patent ownership?",
    f"CR4 = <b>{m['CR4 (%)']:.0f}%</b>, CR8 = {m['CR8 (%)']:.0f}%, HH = {m['HH']:.3f} (equivalent to <b>{m['1/HH']:.0f}</b> equal-sized applicants), Gini = {m['Gini']:.2f}.",
    f"{int(m['firms']):,} applicants and {int(m['patents (sum over firms)']):,} family-applicant links up to {LAST_COMPLETE_YEAR}; name method: {NAME_METHOD}. "
    "Compare with the firms-only column and with the robustness table before writing the interpretation.",
)

## 6. Consolidating or not?

HH, 1/HH and CR4 computed on consecutive windows of `WINDOW` years (counting back from `LAST_COMPLETE_YEAR`). Early windows with very few families are dropped (`MIN_FAMILIES_PER_WINDOW`):
with few families HH is mechanically high, which looks like concentration but is just small numbers.

In [ ]:
def window_table(links_df, last_year, width, min_families):
    rows = []
    end = last_year
    while end - width + 1 >= YEAR_MIN:
        start = end - width + 1
        w = links_df[links_df["year"].between(start, end)]
        fpa = w.groupby("applicant")["unit_id"].nunique().to_numpy()
        if len(fpa) and w["unit_id"].nunique() >= min_families:
            rows.append({"window": f"{start}-{end}", "mid year": (start + end) / 2, "families": w["unit_id"].nunique(),
                         "applicants": len(fpa), "HH": herfindahl(fpa), "1/HH": 1 / herfindahl(fpa),
                         "CR4 (%)": concentration_ratio(fpa, 4)})
        end -= width
    return pd.DataFrame(rows).iloc[::-1].set_index("window")


windows = window_table(links, LAST_COMPLETE_YEAR, WINDOW, MIN_FAMILIES_PER_WINDOW)
save_table(windows.round(4), "concentration_windows")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
windows["HH"].plot(ax=axes[0], marker="o"); axes[0].set(title="Herfindahl index", xlabel="window")
windows["CR4 (%)"].plot(ax=axes[1], marker="o"); axes[1].set(title="CR4 (%)", xlabel="window")
fig.suptitle(f"{SECTOR_NAME}: concentration over time (families per applicant)")
plt.tight_layout()
save_fig(fig, "concentration_windows")
plt.show()
windows.round(3)

In [ ]:
if len(windows) >= 2:
    first_w, last_w = windows.iloc[0], windows.iloc[-1]
    d_hh, d_cr4 = last_w["HH"] - first_w["HH"], last_w["CR4 (%)"] - first_w["CR4 (%)"]
    if d_hh > 0 and d_cr4 > 0:
        verdict = "consolidating (HH and CR4 both higher)"
    elif d_hh < 0 and d_cr4 < 0:
        verdict = "fragmenting / opening up (HH and CR4 both lower)"
    else:
        verdict = "no clear trend (HH and CR4 move in different directions)"
    answer_box(
        "Is the sector consolidating?",
        f"<b>{verdict}</b>.",
        f"From {windows.index[0]} to {windows.index[-1]}: HH {first_w['HH']:.3f} -> {last_w['HH']:.3f}, CR4 {first_w['CR4 (%)']:.0f}% -> {last_w['CR4 (%)']:.0f}%, "
        f"applicants {int(first_w['applicants'])} -> {int(last_w['applicants'])}. Check the windows in between, not only the two ends.",
    )
else:
    print("Fewer than 2 windows with enough families: lower MIN_FAMILIES_PER_WINDOW or WINDOW.")

## 7. Incumbents and new entrants

In [ ]:
by_applicant = links.groupby("applicant")
top_applicants = pd.DataFrame({
    "type": by_applicant["type"].first(),
    "country": by_applicant["ctry"].first(),
    "families": by_applicant["unit_id"].nunique(),
    f"families since {SPLIT_YEAR}": links[links["year"] >= SPLIT_YEAR].groupby("applicant")["unit_id"].nunique(),
    "first year in the sector": by_applicant["year"].min(),
    "last year in the sector": by_applicant["year"].max(),
    "first patent ever": first_ever,
}).dropna(subset=["families"]).fillna({f"families since {SPLIT_YEAR}": 0})
top_applicants["status"] = np.where(top_applicants["first year in the sector"] < SPLIT_YEAR, "incumbent", "entrant")
top_applicants = top_applicants.sort_values("families", ascending=False)

# raw names merged into each applicant: this is what you check by hand to build the alias table
variants = labelled.groupby("applicant")["psn_name"].agg(lambda s: " | ".join(sorted(set(s.dropna()))[:6]))
review = top_applicants.head(100).join(variants.rename("PATSTAT names merged (first 6)"))
save_table(top_applicants, "top_applicants_all")
save_table(review, "top_applicants_for_manual_review")

top_applicants.head(TOP_N)

In [ ]:
first_year = by_applicant["year"].min()      # first family of each applicant in the sector
last_year = by_applicant["year"].max()       # last family of each applicant in the sector

active = links[["applicant", "year"]].drop_duplicates()
active["new to the sector"] = active["year"] == active["applicant"].map(first_year)
active["new to patenting"] = active["year"] == active["applicant"].map(first_ever)
active["last family"] = active["year"] == active["applicant"].map(last_year)

entry_exit = active.groupby("year").agg(
    active_applicants=("applicant", "count"),
    new_to_the_sector=("new to the sector", "sum"),
    new_to_patenting=("new to patenting", "sum"),
    leaving=("last family", "sum"),
)
entry_exit["rate of disappearance (%)"] = entry_exit["leaving"] / entry_exit["active_applicants"] * 100
entry_exit.loc[entry_exit.index > LAST_COMPLETE_YEAR - 3, "rate of disappearance (%)"] = np.nan   # not yet observable

# share of each year's families with at least one applicant new to the sector
links_new = links.assign(new=links["year"] == links["applicant"].map(first_year))
entry_exit["entrants' share of families (%)"] = links_new.groupby(["year", "unit_id"])["new"].any().groupby("year").mean() * 100
save_table(entry_exit.round(2), "entry_exit")
entry_exit.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
entry_exit[["active_applicants", "new_to_the_sector", "new_to_patenting"]].plot(ax=axes[0], marker="o", ms=3)
axes[0].set(title="applicants per year", xlabel="year")
entry_exit[["entrants' share of families (%)", "rate of disappearance (%)"]].plot(ax=axes[1], marker="o", ms=3)
axes[1].set(title="entry and exit", xlabel="year", ylim=(0, 100))
axes[1].yaxis.set_major_formatter(PercentFormatter(100))
for ax in axes:
    ax.axvspan(LAST_COMPLETE_YEAR + 0.5, YEAR_MAX + 0.5, color="grey", alpha=0.15)      # incomplete years
    for yr, label in EVENTS.items():                                                      # external events (PEST section)
        ax.axvline(yr, color="firebrick", lw=0.8, ls=":")
        ax.annotate(label, (yr, ax.get_ylim()[1]), rotation=90, va="top", ha="right", fontsize=7, color="firebrick")
fig.suptitle(f"{SECTOR_NAME}: entry and exit")
plt.tight_layout()
save_fig(fig, "entry_exit")
plt.show()

In [ ]:
top = top_applicants.head(TOP_N)
incumbents = top[top["status"] == "incumbent"].index.tolist()
entrants = top[top["status"] == "entrant"].index.tolist()
recent_entry = entry_exit.loc[LAST_COMPLETE_YEAR - 4:LAST_COMPLETE_YEAR, "entrants' share of families (%)"].mean()

answer_box(
    "Who are the incumbents, and who are the new entrants?",
    f"Among the {TOP_N} largest applicants, <b>{len(incumbents)} are incumbents</b> and <b>{len(entrants)} are entrants</b>.",
    f"Largest incumbents: {', '.join(incumbents[:3]) or 'none'}. "
    f"Largest entrants (first family from {SPLIT_YEAR}): {', '.join(entrants[:3]) or 'none'}. "
    f"In the last five complete years, on average {recent_entry:.0f}% of each year's families had at least one applicant new to the sector.",
)

## 8. Which subfields are growing faster?

Subfields are CPC groups cut at two digits after the slash (`G06N10/40`). The older codes `G06N99/002` and `Y10S977/933` have no finer structure: they stay as one **legacy** bucket each and are not comparable with the `G06N10/..` subfields.

In [ ]:
RECENT = (LAST_COMPLETE_YEAR - 4, LAST_COMPLETE_YEAR)          # the last 5 complete years
PAST = (RECENT[0] - 5, RECENT[0] - 1)                          # the 5 years before


def subfield(cpc):
    "CPC group with the first two digits after the slash: G06N10/4 -> G06N10/40 style; legacy codes stay whole."
    if "/" not in cpc:
        return cpc
    main_group, subgroup = cpc.split("/", 1)
    if cpc.startswith(("G06N99/002", "Y10S977/933")):
        return f"{main_group}/{subgroup} (legacy)"
    return f"{main_group}/{subgroup[:2]}"


in_sector = family_cpc[family_cpc["cpc"].str.startswith(tuple(SECTOR_CPC_PREFIXES))].copy()
in_sector["subfield"] = in_sector["cpc"].map(subfield)
in_sector = in_sector.drop_duplicates(["unit_id", "subfield"])
print(f"{in_sector['unit_id'].nunique():,} of {family_cpc['unit_id'].nunique():,} families carry a sector CPC code "
      "(the others entered Step 0 through IPC or keywords only and have no subfield here)")

past = in_sector[in_sector["year"].between(*PAST)].groupby("subfield")["unit_id"].nunique()
recent = in_sector[in_sector["year"].between(*RECENT)].groupby("subfield")["unit_id"].nunique()
subfields = pd.DataFrame({f"families {PAST[0]}-{PAST[1]}": past, f"families {RECENT[0]}-{RECENT[1]}": recent}).fillna(0)
subfields["growth (%)"] = (subfields.iloc[:, 1] / subfields.iloc[:, 0].replace(0, np.nan) - 1) * 100
subfields = subfields.sort_values(subfields.columns[1], ascending=False)
save_table(subfields.round(1), "subfields_growth")
subfields.round(1)

In [ ]:
plot = subfields[(subfields.iloc[:, 0] > 0) & ~subfields.index.str.contains("legacy")]   # growth needs a non-zero base
if plot.empty:
    print("No subfield with families in both periods: nothing to plot.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(plot.iloc[:, 1], plot["growth (%)"], s=80, alpha=0.6)
    for name, row in plot.iterrows():
        ax.annotate(name, (row.iloc[1], row["growth (%)"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set(title=f"{SECTOR_NAME}: subfield growth map", xlabel=f"families {RECENT[0]}-{RECENT[1]} (size today)",
           ylabel=f"growth vs {PAST[0]}-{PAST[1]}")
    ax.yaxis.set_major_formatter(PercentFormatter(100))
    save_fig(fig, "subfield_growth_map")
    plt.show()

In [ ]:
if not plot.empty:
    size_today = plot.iloc[:, 1]
    fastest = plot[size_today >= size_today.median()].sort_values("growth (%)", ascending=False).head(3)
    answer_box(
        "Which subfields are growing fastest?",
        "<b>" + ", ".join(f"{name} ({row['growth (%)']:+.0f}%)" for name, row in fastest.iterrows()) + "</b>",
        f"Growth of families from {PAST[0]}-{PAST[1]} to {RECENT[0]}-{RECENT[1]}, among the subfields at least as large as the median one. "
        "Check the titles of these CPC groups in the CPC scheme.",
    )

## 9. External forces (PEST)

This question is answered with sources **outside PATSTAT**; the patent data only tells you *when* things changed. Suggested use:
1. For each dimension collect 2-4 dated facts with a source (policy documents, funding announcements, company reports, news).
2. Put the dated events in `EVENTS` in the config: they appear as vertical lines on the entry/exit chart, so you can see whether entry or applicant counts move after them (association, not proof).
3. Say for each factor whether it should push towards concentration (high entry barriers, scale needed, capital) or towards entry (public funding, open research, spin-offs).

| dimension | what to look for in quantum computing |
|---|---|
| **P**olitical | national quantum strategies and funding programmes, export controls, public-private programmes |
| **E**conomic | venture capital and public listings of quantum start-ups, big-tech R&D budgets, cost of hardware |
| **S**ocial | talent supply (PhDs, university programmes), public perception and security concerns |
| **T**echnological | qubit technologies (superconducting, trapped ion, photonic ...), error correction milestones, cloud access |

In [ ]:
pest = pd.DataFrame(columns=["dimension", "factor", "date", "source", "expected effect on concentration / entry"])
# add rows as you collect evidence, e.g.
# pest.loc[len(pest)] = ["Political", "<programme name>", 2018, "<link>", "more entry of universities and start-ups"]
pest

## 10. Files saved

In [ ]:
meta = {
    "run date": str(date.today()), "ENV": ENV, "sector file": str(SECTOR_FILE), "SECTOR_RULE": SECTOR_RULE,
    "NAME_METHOD": NAME_METHOD, "EMBED": EMBED, "TERR_PARAMS": TERR_PARAMS, "manual aliases": len(ALIASES),
    "years": [YEAR_MIN, YEAR_MAX], "LAST_COMPLETE_YEAR": LAST_COMPLETE_YEAR, "SPLIT_YEAR": SPLIT_YEAR, "WINDOW": WINDOW,
    "families": int(links["unit_id"].nunique()), "applicants": int(links["applicant"].nunique()),
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(meta, indent=2, default=str))

try:                                           # the harmonised family-applicant table is the input of the next steps
    links.to_parquet(OUTPUT_DIR / "links_family_applicant.parquet", index=False)
except Exception:                              # no parquet engine installed
    links.to_csv(OUTPUT_DIR / "links_family_applicant.csv", index=False)

print("everything is in:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print("  ", p.name)